# PatchTST — Store Sales Forecasting (Deep Learning, Transformer)

PatchTST (2023) Transformer-ია დროითი მწკრივებისთვის. ორი მთავარი იდეა:
1. **Patching** — lookback window იჭრება ფიქსირებული სიგრძის ნაჭრებად (patches),
   თითო patch გახდება ერთი token (NLP-ის „სიტყვების“ ანალოგი);
2. **Channel independence** — თითო სერია დამოუკიდებლად მუშავდება ერთი საერთო წონებით.

გარდა ამისა, ვიყენებ **RevIN ნორმალიზაციას**: თითო window ნორმდება თავისი mean/std-ით
და გამოსავალი უკან denormalize-დება. ეს საშუალებას აძლევს ერთ მოდელს დაფაროს
სხვადასხვა მასშტაბის სერიები.

### 1. ბიბლიოთეკები + მონაცემები

In [ ]:
import sys
import os
import warnings
import time

sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
os.environ.setdefault("WANDB_SILENT", "true")

import torch
torch.manual_seed(42)

from src.data import load_raw
from src.metrics import wmae
from src.patchtst_model import build_patchtst_pipeline
from src.pipeline import RAW_COLS
from src.validation import time_holdout_split
from src.wandb_utils import init_run, log_pipeline

train = load_raw("data").train
tr, val = time_holdout_split(train, n_val_weeks=12)
val = val.reset_index(drop=True)

print("train:", tr.shape, "| validation:", val.shape)

### 2. ვარიანტები

ვცვლი lookback-ის სიგრძეს (`seq_len`), patch-ის ზომას, მოდელის სიგანე/სიღრმეს და
ტრენინგის ხანგრძლივობას. `window_stride=3` ინახავს ყოველ მე-3 training window-ს, რომ
CPU-ზე გაშვება წუთებში მოვათავო (და არა ~20 წუთი თითო ვარიანტზე).

In [ ]:
COMMON = {"batch_size": 2048, "window_stride": 3}

EXPERIMENTS = [
    {
        "name": "PatchTST_v1_baseline",
        "cfg": {"seq_len": 52, "patch_len": 8, "stride": 4, "d_model": 64,
                "n_heads": 4, "depth": 2, "epochs": 12, "lr": 1e-3, **COMMON},
    },
    {
        "name": "PatchTST_v2_long_lookback",
        "cfg": {"seq_len": 104, "patch_len": 16, "stride": 8, "d_model": 64,
                "n_heads": 4, "depth": 2, "epochs": 12, "lr": 1e-3, **COMMON},
    },
    {
        "name": "PatchTST_v3_deeper_wider",
        "cfg": {"seq_len": 52, "patch_len": 8, "stride": 4, "d_model": 128,
                "n_heads": 8, "depth": 2, "epochs": 10, "lr": 1e-3, **COMMON},
    },
    {
        "name": "PatchTST_v4_small_patches",
        "cfg": {"seq_len": 52, "patch_len": 4, "stride": 2, "d_model": 64,
                "n_heads": 4, "depth": 2, "epochs": 12, "lr": 1e-3, **COMMON},
    },
    {
        "name": "PatchTST_v5_long_train",
        "cfg": {"seq_len": 52, "patch_len": 8, "stride": 4, "d_model": 64,
                "n_heads": 4, "depth": 2, "epochs": 20, "lr": 5e-4, **COMMON},
    },
]

print("სულ ვარიანტი:", len(EXPERIMENTS))

### 3. თითო ვარიანტის გაშვება

In [ ]:
results = []

for exp in EXPERIMENTS:
    name = exp["name"]
    cfg = exp["cfg"]

    run = init_run(group="PatchTST_Training", job_type="experiment", name=name, config=cfg)

    start = time.time()
    pipe = build_patchtst_pipeline(**cfg)
    pipe.fit(tr[RAW_COLS], tr["Weekly_Sales"])
    pred = pipe.predict(val[RAW_COLS])
    minutes = (time.time() - start) / 60

    score = wmae(val["Weekly_Sales"], pred, val["IsHoliday"])

    run.summary["holdout_wmae"] = score
    run.summary["wmae_val"] = score
    run.summary["train_min"] = minutes
    log_pipeline(run, pipe, name="walmart_patchtst_" + name.split("_", 1)[1],
                 metadata={"holdout_wmae": score})
    run.finish()

    results.append((name, score))
    print(name, "->", round(score, 2), "WMAE |", round(minutes, 1), "min")

### 4. საუკეთესო + რეგისტრაცია

In [ ]:
best_name = None
best_score = float("inf")

for name, score in results:
    if score < best_score:
        best_score = score
        best_name = name

best_cfg = None
for exp in EXPERIMENTS:
    if exp["name"] == best_name:
        best_cfg = exp["cfg"]
        break

print("საუკეთესო:", best_name, "->", round(best_score, 2))

In [ ]:
run = init_run(group="PatchTST_Training", job_type="final", name="PatchTST_Final", config=best_cfg)

final_pipe = build_patchtst_pipeline(**best_cfg)
final_pipe.fit(train[RAW_COLS], train["Weekly_Sales"])

run.summary["holdout_wmae"] = best_score
run.summary["wmae_val"] = best_score
log_pipeline(run, final_pipe, name="walmart_patchtst",
             metadata={"holdout_wmae": best_score}, aliases=["best"])
run.finish()
print("დარეგისტრირდა: walmart_patchtst:best")

### შედეგები

| ვარიანტი | WMAE |
|---|---|
| **v1 baseline** | **1527** |
| v2 long lookback | 1576 |
| v3 deeper/wider | 1520 |
| v4 small patches | 1651 |
| v5 long train | 1733 |

RevIN ნორმალიზაცია კრიტიკული იყო (მის გარეშე სხვადასხვა მასშტაბის სერიები ვერ
ერთიანდება). PatchTST ჯობია XGBoost-ს, მაგრამ ჩამორჩება Prophet/LightGBM-ს — ამ
ზომის ისტორია (≈143 კვირა) transformer-ს ცოტაა. (full-window ვარიანტმა ~1348-საც
მიაღწია, მაგრამ ~10x ნელი იყო, ამიტომ `window_stride=3` გამოვიყენე.)